# Dia 7 — Seguindo o dinheiro 💸

**Linux para OSINT — 21 dias** · baseado no curso de [cipher387](https://github.com/cipher387/linux-for-OSINT-21-day) (MIT)

📘 Texto completo do dia: [dia-07-sed-e-awk](https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias/tree/main/dias/dia-07-sed-e-awk)

## 🎬 Missão

> Chegou a resposta do ofício: um extrato com 240 transferências saindo das
> contas das vítimas em março, e uma lista de "apelidos" que o painel da
> PixVelox dava para as contas de destino.
>
> — A conta que mais aparece é a CT-0450 — você arrisca.
>
> Bento nem levanta a cabeça.
> — Aparecer muito não é receber muito. Some os valores. Por conta. E tira
> os estornos, que esse dinheiro voltou. Isso é **uma** linha de `awk`.
>
> Mestra Iara completa:
> — E quando achar a conta, limpe o apelido dela. Exportação de painel
> sempre vem suja.

**Sua tarefa:** descobrir qual conta de destino **recebeu mais dinheiro** em
PIX e limpar o apelido dela em `desafio/contas.txt`. A flag é
`VAZANTE{apelido_limpo}`.

## ⚙️ Preparação

Rode a célula abaixo **uma vez**. Ela baixa o repositório para dentro do Colab e define a variável `L4O_HOME`, que todas as outras células usam.

> No Colab você é `root` e o `apt` funciona. Cada célula `%%bash` roda num processo separado: variáveis e `cd` não passam de uma célula para outra.

In [ ]:
import os, subprocess
from pathlib import Path

# Se já estamos dentro do repositório (Codespaces, máquina local), usa ele.
aqui = Path.cwd()
candidatos = [aqui, *aqui.parents]
raiz = next((p for p in candidatos if (p / 'lib' / 'comum.sh').exists()), None)

if raiz is None:  # Colab: clona o repositório
    raiz = Path('/content/linux-para-osint-21-dias')
    if not raiz.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias', str(raiz)], check=True)

os.environ['L4O_HOME'] = str(raiz)
os.environ['L4O_CORES'] = '1'
print('Repositório em', raiz)

### sed: substituir texto (s/velho/novo/)

```text
sed = "stream editor": lê o texto linha a linha e aplica uma edição.
O arquivo original NÃO muda — o resultado sai na tela.
s/Normal/SUSPEITO/ troca a PRIMEIRA ocorrência de cada linha.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-07-sed-e-awk" && source "$L4O_HOME/lib/comum.sh"
titulo "sed s///"
head -n 4 desafio/relatorio.txt | sed 's/Normal/SUSPEITO/'

### Trocar todas as ocorrências: /g

```text
Sem o g, só a primeira de cada linha muda. Com g (global), todas.
Aqui trocamos cada dígito por "#": [0-9] significa "qualquer dígito".
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-07-sed-e-awk" && source "$L4O_HOME/lib/comum.sh"
titulo "sed s///g"
sed 's/[0-9]/#/g' desafio/digitos.txt
echo "--- sem o g (só o primeiro dígito de cada linha):"
sed 's/[0-9]/#/' desafio/digitos.txt

### Imprimir linhas específicas: -n Np

```text
-n desliga a impressão automática; p imprime só o que você pediu.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-07-sed-e-awk" && source "$L4O_HOME/lib/comum.sh"
titulo "sed -n: linha 7 e intervalo 5 a 9"
sed -n 7p desafio/relatorio.txt
echo "---"
sed -n 5,9p desafio/relatorio.txt

### Substituir só num intervalo de linhas

```text
'1,3 s/a/b/g' aplica a troca só nas linhas 1 a 3.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-07-sed-e-awk" && source "$L4O_HOME/lib/comum.sh"
titulo "sed com intervalo"
sed '1,3 s/Normal/CONFERIDO/g' desafio/relatorio.txt | head -n 6

### Apagar linhas e editar uma cópia no lugar

```text
  sed '/padrão/d'  apaga as linhas que batem
  sed -i           edita o ARQUIVO (use numa cópia!)
  -e               várias edições em sequência
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-07-sed-e-awk" && source "$L4O_HOME/lib/comum.sh"
titulo "sed d e sed -i (numa cópia)"
sed '/^linha/d' desafio/relatorio.txt
mkdir -p treino
cp desafio/relatorio.txt treino/
sed -i -e 's/Normal/OK/g' -e '/^linha 1[0-4]/d' treino/relatorio.txt
tail -n 3 treino/relatorio.txt
rm -r treino

### Limpando dados sujos com sed

```text
Dados exportados vêm com lixo: espaços, marcadores, separadores estranhos.
^ = começo da linha, $ = fim, [[:space:]]* = qualquer quantidade de espaços.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-07-sed-e-awk" && source "$L4O_HOME/lib/comum.sh"
titulo "Limpando contas.txt"
sed -e '/^#/d' -e 's/[[:space:]]*##fim$//' desafio/contas.txt | head -n 3

### awk: imprimir linhas ($0) e numerá-las (NR)

```text
awk lê linha a linha e divide cada uma em CAMPOS ($1, $2, ...).
$0 é a linha inteira; NR é o número da linha.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-07-sed-e-awk" && source "$L4O_HOME/lib/comum.sh"
titulo "awk \$0 e NR"
awk '{print $0}' desafio/vitimas.csv
echo "---"
awk '{print NR, $0}' desafio/vitimas.csv

### Escolhendo colunas: -F

```text
-F, diz que o separador de campos é a vírgula (CSV).
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-07-sed-e-awk" && source "$L4O_HOME/lib/comum.sh"
titulo "awk -F, colunas 1 e 3"
awk -F, '{print $1, $3}' desafio/vitimas.csv
echo "--- primeiras linhas do extrato (data, destino, valor):"
awk -F, 'NR <= 5 {print $1, $3, $4}' desafio/extrato.csv

### Filtrando linhas

```text
  /^v/        linhas que começam com "v"
  $3 > 1000   linhas cujo 3º campo é maior que 1000
  $5 == "ESTORNO"   comparação de texto
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-07-sed-e-awk" && source "$L4O_HOME/lib/comum.sh"
titulo "awk com filtros"
awk '/^v/' desafio/vitimas.csv
echo "--- prejuízo acima de R\$ 1.000:"
awk -F, 'NR > 1 && $3 > 1000 {print $1, $3}' desafio/vitimas.csv
echo "--- estornos no extrato:"
awk -F, '$5 == "ESTORNO"' desafio/extrato.csv | head -n 3

### Somando com awk

```text
Variáveis do awk começam em zero. O bloco END roda depois da última linha.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-07-sed-e-awk" && source "$L4O_HOME/lib/comum.sh"
titulo "Somas"
awk -F, 'NR > 1 {total += $3} END {print "Prejuízo total:", total}' desafio/vitimas.csv
awk -F, 'NR > 1 && $5 == "PIX" {n++; total += $4} END {printf "PIX: %d transações, R$ %.2f\n", n, total}' desafio/extrato.csv

### Somando POR GRUPO: arrays associativos

```text
soma[$2] += $3 cria uma "gaveta" para cada valor do campo 2 e vai somando.
É o "GROUP BY" do terminal.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-07-sed-e-awk" && source "$L4O_HOME/lib/comum.sh"
titulo "Prejuízo por cidade"
awk -F, 'NR > 1 {soma[$2] += $3} END {for (c in soma) print soma[c], c}' desafio/vitimas.csv | sort -rn

### Contar não é somar

```text
Quantas vezes cada conta de destino aparece no extrato?
(Compare depois com a SOMA dos valores — é o desafio de hoje.)
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-07-sed-e-awk" && source "$L4O_HOME/lib/comum.sh"
titulo "Quantas transferências por conta de destino"
awk -F, 'NR > 1 {n[$3]++} END {for (c in n) print n[c], c}' desafio/extrato.csv | sort -rn

### sed + awk juntos

```text
O pipe deixa cada ferramenta fazer o que faz melhor: sed limpa, awk calcula.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-07-sed-e-awk" && source "$L4O_HOME/lib/comum.sh"
titulo "sed limpa, awk formata"
sed -e '/^#/d' -e 's/[[:space:]]*##fim$//' -e 's/~/ /g' desafio/contas.txt \
  | awk -F' ; *' '{printf "%-8s -> %s\n", $1, $2}'
dica "Agora some os valores do extrato por conta de destino. Quem recebeu mais?"

## 🐍 Ponte Python

O mesmo resultado só com a biblioteca padrão do Python:

In [ ]:
!cd "$L4O_HOME/dias/dia-07-sed-e-awk" && python3 equivalente.py

## 🧩 Desafio

Encontrou a flag? Troque o texto abaixo e rode a célula.

Dicas e passo a passo estão no README do dia.

In [ ]:
flag = 'VAZANTE{troque_aqui}'  # ✏️ cole sua resposta
!cd "$L4O_HOME/dias/dia-07-sed-e-awk" && bash checar.sh "{flag}"

---

📝 Anote o que aprendeu no `DIARIO.md` do repositório.

Feito por [Deivison Lourenço](https://github.com/Ridd1kulusC0d3r) · curso original de [cipher387 / Cyber Detective](https://cybdetective.com)